# Demo: Cross-Model Hallucination Detection from Output-Side Signals

**3 examples**:
- one low-risk correct answer
- two high-risk incorrect answer

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DEMO_DIR = Path("demo_files")

CHECKPOINT_PATH = DEMO_DIR / "mlp_checkpoint_mistral_to_gemma.pt"
EXAMPLES_PATH = DEMO_DIR / "demo_examples.csv"

FEATURES = [
    "entropy",
    "margin",
    "length_norm_logprob",
    "selected_num_tokens",
]

THRESHOLD = 0.5

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),

            
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

model = MLP(
    input_dim=len(ckpt["features"]),
    hidden_dims=tuple(ckpt.get("hidden_dims", (128, 64))),
    dropout=float(ckpt.get("dropout", 0.1)),
).to(DEVICE)

model.load_state_dict(ckpt["model"])
model.eval()

scaler = StandardScaler()
scaler.mean_ = ckpt["scaler_mean"]
scaler.scale_ = ckpt["scaler_scale"]
scaler.var_ = scaler.scale_ ** 2
scaler.n_features_in_ = len(ckpt["features"])

feature_names = ckpt["features"]

print("Checkpoint feature order:", feature_names)

def parse_json_list(x):
    if isinstance(x, list):
        return x
    
    return json.loads(x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def format_choices(row):
    labels = parse_json_list(row["choice_labels"])
    texts = parse_json_list(row["choice_texts"])

    return "\n".join([f"{lab}. {txt}" for lab, txt in zip(labels, texts)])

def score_mlp_from_row(row):
    x = row[feature_names].astype(float).to_numpy().reshape(1, -1)
    x_scaled = scaler.transform(x)
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=DEVICE)

    with torch.no_grad():
        logit = model(x_t).item()

    prob = float(sigmoid(logit))
    pred = int(prob >= THRESHOLD)
    
    return prob, pred

In [ ]:
demo_df = pd.read_csv(EXAMPLES_PATH)

display(demo_df[[
    "demo_name",
    "source_model",
    "correct",
    "pred_label",
    "gold_label"
]])

In [ ]:
def run_demo_example(row):
    print(row['demo_name'])

    print("=" * 90)
    print("QUESTION:")
    print(row["question"])
    print()
    print("CHOICES:")
    print(format_choices(row))
    print()
    print(f"GENERATION SOURCE: {row['source_model']}")
    print(f"PREDICTED ANSWER:  {row['pred_label']}")
    print(f"GOLD ANSWER:       {row['gold_label']}")
    print(f"CORRECT:           {bool(int(row['correct']))}")
    print()

    print("EXTRACTED FEATURES:")
    print(f"  entropy               = {row['entropy']:.4f}")
    print(f"  margin                = {row['margin']:.4f}")
    print(f"  length_norm_logprob   = {row['length_norm_logprob']:.4f}")
    print(f"  selected_num_tokens   = {int(row['selected_num_tokens'])}")
    print()

    mlp_prob, mlp_pred = score_mlp_from_row(row)
    print(f"MLP HALLUCINATION-RISK SCORE: {mlp_prob:.4f} -> flagged={bool(mlp_pred)}")
    print("=" * 90)

##### QUESTION:
Where could a sloth live?

##### CHOICES:
A. tropical jungle,
B. manual,
C. work,
D. transit,
E. countryside

In [ ]:
# Example 1: low-risk correct answer
row = demo_df.iloc[0]
run_demo_example(row)

##### QUESTION:
John needed a straight wire.  Unfortunately, this one had endured some abuse and had become what?

##### CHOICES:
A. bent,
B. warped,
C. crooked,
D. straightforth,
E. curved

In [ ]:
# Example 2: high-risk incorrect answer
row = demo_df.iloc[1]
run_demo_example(row)

##### QUESTION:
What is the primary purpose of cars?

##### CHOICES:
A. cost money,
B. slow down,
C. move people,
D. turn right,
E. get girls


In [ ]:
# Example 3: high-risk incorrect answer
row = demo_df.iloc[2]
run_demo_example(row)

In [ ]:
summary_rows = []
for _, row in demo_df.iloc[:3].iterrows():
    mlp_prob, mlp_pred = score_mlp_from_row(row)

    summary_rows.append({
        "demo_example": row["demo_name"],
        "source_generation": row["source_model"],
        "pred_label": row["pred_label"],
        "gold_label": row["gold_label"],
        "correct": int(row["correct"]),
        "mlp_risk": mlp_prob,
        "mlp_flag": mlp_pred,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)